In [5]:
# =============================================================================
# NOTEBOOK: 015_agent1_5_process_graph.ipynb
# Automating Interview-Based Generative-AI ROI Measurement
#   — A Domain-Agnostic Three-Agent Pipeline (DSRM Design Artifact)
#
# AGENT 1.5 — Process-Graph Construction & AS-IS/TO-BE Derivation
#   Input : artifacts/inference/agent1_work_units.json  (46 graded work units)
#   Method: (a) assign a flowchart SHAPE TYPE to each unit (start/task/decision/end)
#           (b) infer sequential + branch EDGES to form the AS-IS process graph
#           (c) assign each unit to an organizational LANE (swimlane)
#           (d) derive the TO-BE graph by applying the automatability rules
#   Output: artifacts/inference/process_graph.json  (as_is + to_be graphs)
#
# Why this stage exists (paper logic, per author's Image 3):
#   The AS-IS vs TO-BE comparison of human-touched activities is the ROI
#   argument. We reconstruct the process as a typed graph so that (i) the
#   downstream effort agent can COUNT human activities on the graph, and (ii)
#   a static swimlane figure can be rendered for the paper.
#
# TO-BE transformation rules (confirmed):
#   full    -> moved to the AI-agent lane; human effort retained = 0%
#   partial -> split into "AI draft" (AI lane) + "human review" (human lane);
#              human effort retained = PARTIAL_RETAIN (default 30%, a parameter)
#   manual  -> unchanged; human effort retained = 100%
#
# All example content and code are in English for journal submission.
# =============================================================================


# %%
# =============================================================================
# Cell 1. Bootstrap foundation from Notebook 00 (self-contained)
# =============================================================================
import os
import re
import json
import time
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import Optional, Any

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_RAW  = ROOT / "data" / "raw"
ARTIFACTS = ROOT / "artifacts"
INFER     = ARTIFACTS / "inference"
TAB       = ARTIFACTS / "tables"
CACHE     = ARTIFACTS / "llm_cache"
for p in (INFER, TAB, CACHE):
    p.mkdir(parents=True, exist_ok=True)


def rel(p) -> str:
    try:
        return str(Path(p).resolve().relative_to(ROOT.resolve()))
    except ValueError:
        return Path(p).name


SEED = 42
np.random.seed(SEED)

RUN_MANIFEST = ARTIFACTS / "run_manifest.json"
if not RUN_MANIFEST.exists():
    raise FileNotFoundError("[ERROR] run_manifest.json missing. Run nb 00 first.")
manifest = json.loads(RUN_MANIFEST.read_text(encoding="utf-8"))

MODELS       = manifest["models"]
DEFAULT_TIER = manifest["default_tier"]
AUTO_GRADES  = manifest["auto_grades"]
MISSING      = manifest["missing_sentinel"]

load_dotenv(ROOT / ".env")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env.")
client = OpenAI(api_key=OPENAI_API_KEY)


# %%
# =============================================================================
# Cell 2. Re-declare cost tracker + llm_call() (identical to nb 00)
# =============================================================================
class CostTracker:
    def __init__(self):
        self.records: list[dict] = []

    def add(self, tier, model, usage, tag=""):
        p = MODELS.get(tier, {})
        pin, pcached, pout = p.get("in"), p.get("cached_in"), p.get("out")
        pt = getattr(usage, "prompt_tokens", 0) or 0
        ct = getattr(usage, "completion_tokens", 0) or 0
        cached = 0
        det = getattr(usage, "prompt_tokens_details", None)
        if det is not None:
            cached = getattr(det, "cached_tokens", 0) or 0
        fresh = max(pt - cached, 0)
        cost = None
        if None not in (pin, pout):
            pc = pcached if pcached is not None else pin
            cost = (fresh * pin + cached * pc + ct * pout) / 1_000_000
        self.records.append({"tag": tag, "tier": tier, "model": model,
                             "prompt_tokens": pt, "cached_tokens": cached,
                             "completion_tokens": ct, "cost_usd": cost})
        return cost or 0.0

    def total_usd(self):
        return float(sum(r["cost_usd"] or 0.0 for r in self.records))


COST = CostTracker()


def _safe_json(text):
    if not text:
        return None
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.S).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    for opener, closer in (("{", "}"), ("[", "]")):
        i, j = t.find(opener), t.rfind(closer)
        if 0 <= i < j:
            try:
                return json.loads(t[i:j + 1])
            except Exception:
                continue
    return None


def _cache_key(model, system, user, temperature, response_json):
    raw = json.dumps({"m": model, "s": system, "u": user,
                      "t": temperature, "j": response_json},
                     ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()[:24]


def llm_call(system, user, tier=DEFAULT_TIER, temperature=0.0,
             response_json=True, tag="", use_cache=True, max_retries=3):
    model = MODELS[tier]["name"]
    key = _cache_key(model, system, user, temperature, response_json)
    cache_file = CACHE / f"{key}.json"
    if use_cache and cache_file.exists():
        c = json.loads(cache_file.read_text(encoding="utf-8"))
        c["cached"] = True
        # Preserve the ORIGINAL (already-paid) cost so the ledger reflects the
        # true analysis cost even on cached re-runs; also re-log it to COST so
        # cumulative spend is complete regardless of cache state.
        original_cost = c.get("cost_usd", 0.0) or 0.0
        COST.records.append({
            "tag": tag or c.get("model", ""), "tier": c.get("tier", tier),
            "model": c.get("model", ""), "prompt_tokens": 0,
            "cached_tokens": 0, "completion_tokens": 0,
            "cost_usd": original_cost,
        })
        c["cost_usd"] = original_cost
        return c
    kwargs: dict[str, Any] = {
        "model": model,
        "messages": [{"role": "system", "content": system},
                     {"role": "user", "content": user}],
    }
    if response_json:
        kwargs["response_format"] = {"type": "json_object"}
    kwargs["temperature"] = temperature
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            resp = client.chat.completions.create(**kwargs)
            text = resp.choices[0].message.content or ""
            parsed = _safe_json(text) if response_json else None
            if response_json and parsed is None:
                raise ValueError("Response was not valid JSON.")
            cost = COST.add(tier, model, resp.usage, tag=tag or model)
            out = {"text": text, "json": parsed, "cached": False,
                   "tier": tier, "model": model, "cost_usd": cost}
            if use_cache:
                cache_file.write_text(json.dumps(out, ensure_ascii=False),
                                      encoding="utf-8")
            return out
        except TypeError as e:
            if "temperature" in kwargs:
                kwargs.pop("temperature", None); last_err = e; continue
            last_err = e
        except Exception as e:
            last_err = e; time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f"[llm_call] failed after {max_retries} retries: {last_err}")


print("[INFO] CostTracker, llm_call() re-established for nb 015.")


# %%
# =============================================================================
# Cell 3. Load Agent-1 output (the graded work units)
# =============================================================================
A1_PATH = INFER / "agent1_work_units.json"
if not A1_PATH.exists():
    raise FileNotFoundError(
        "[ERROR] agent1_work_units.json missing. Run 01_agent1_task_extraction first."
    )
a1 = json.loads(A1_PATH.read_text(encoding="utf-8"))
units = a1["work_units"]
print(f"[INFO] Loaded {len(units)} work units from {rel(A1_PATH)}")

# Ordered list of lanes (organizational actors) as they first appear.
# Lanes are derived from the data — no domain-specific lane names are hardcoded.
lane_order: list[str] = []
for u in units:
    actor = u.get("actor") or MISSING
    if actor not in lane_order:
        lane_order.append(actor)
print(f"[INFO] Derived {len(lane_order)} organizational lanes: {lane_order}")


# %%
# =============================================================================
# Cell 4. Agent-1.5 prompt — assign shape TYPE + branch edges (domain-agnostic)
#
# The model sees only the ordered work units (id, name, actor, grade) and must:
#   * assign each unit a flowchart shape type: start | task | decision | end
#   * declare edges (from_id -> to_id), including a branch label for decisions
# This mirrors the author's legend (Image 1): start/end hexagons, activity
# rectangles, decision diamonds, and directional flow. Nothing here references
# the specific business domain.
# =============================================================================
compact_units = [
    {"id": u["id"], "name": u["name"],
     "actor": u.get("actor", MISSING), "grade": u.get("auto_grade", MISSING)}
    for u in units
]

AGENT15_SYSTEM = """\
You are Agent 1.5 (Process-Graph Construction) in an ROI-estimation pipeline.
You receive an ORDERED list of work units already extracted from an operational
interview (any business domain). Your job is to turn them into a process graph.

For every unit, assign exactly one flowchart SHAPE TYPE:
  - "start"    : the entry point that triggers the process (usually the first unit)
  - "end"      : a terminal unit that dispatches/closes the process
  - "decision" : a unit that branches on a condition (yes/no, new/re-run, pass/fail)
  - "task"     : any ordinary activity that is neither start, end, nor a branch

Then declare directed EDGES connecting the units in the order the process flows.
  - Normal flow: {"from": "wX", "to": "wY", "label": ""}
  - Decision branches: emit two edges from the decision unit, each with a short
    branch label, e.g. "yes"/"no" or "new"/"re-run".
  - The graph should be a single connected flow from the start unit to the end
    unit(s). Keep it mostly linear; add branches only where the units imply them.

Rules:
  - Use ONLY the given unit ids. Do not invent units.
  - Exactly one "start". At least one "end".
  - Do not change names or grades; you only add "type" and the edge list.

Return ONLY JSON:
{
  "nodes": [ {"id": "w1", "type": "start|task|decision|end"}, ... ],
  "edges": [ {"from": "wX", "to": "wY", "label": ""}, ... ]
}
"""

AGENT15_USER = (
    "Ordered work units (id, name, actor, grade):\n"
    + json.dumps(compact_units, ensure_ascii=False, indent=1)
)

print(f"[INFO] Agent-1.5 prompt ready (system {len(AGENT15_SYSTEM)} chars, "
      f"user {len(AGENT15_USER)} chars).")


# %%
# =============================================================================
# Cell 5. Run Agent 1.5 (deterministic) and validate the returned graph
# =============================================================================
res = llm_call(
    system=AGENT15_SYSTEM, user=AGENT15_USER,
    tier=DEFAULT_TIER, temperature=0.0,
    response_json=True, tag="agent15_graph",
)
graph = res["json"] or {}
nodes_raw = graph.get("nodes", [])
edges_raw = graph.get("edges", [])
print(f"[INFO] Agent 1.5 returned {len(nodes_raw)} nodes, {len(edges_raw)} edges "
      f"(cached={res['cached']}, cost=${res['cost_usd']:.5f}).")

# --- Validation gate: coerce types, guarantee ids, repair start/end ----------
VALID_TYPES = {"start", "task", "decision", "end"}
id2unit = {u["id"]: u for u in units}
type_by_id: dict[str, str] = {}

for n in nodes_raw:
    nid = n.get("id")
    t = str(n.get("type", "task")).strip().lower()
    if nid in id2unit and t in VALID_TYPES:
        type_by_id[nid] = t

# Any unit the model omitted defaults to "task".
for u in units:
    type_by_id.setdefault(u["id"], "task")

# Guarantee exactly one start and at least one end using unit order as fallback.
ordered_ids = [u["id"] for u in units]
if "start" not in type_by_id.values():
    type_by_id[ordered_ids[0]] = "start"
if "end" not in type_by_id.values():
    type_by_id[ordered_ids[-1]] = "end"

# Keep only edges between known ids.
edges = [
    {"from": e["from"], "to": e["to"], "label": str(e.get("label", "")).strip()}
    for e in edges_raw
    if e.get("from") in id2unit and e.get("to") in id2unit
]

# If the model produced too few edges, fall back to a linear chain so the graph
# is always connected (a safe, transparent default).
if len(edges) < len(units) - 1:
    print("[WARN] Sparse edge set; adding a linear backbone as fallback.")
    have = {(e["from"], e["to"]) for e in edges}
    for a, b in zip(ordered_ids, ordered_ids[1:]):
        if (a, b) not in have:
            edges.append({"from": a, "to": b, "label": ""})

from collections import Counter
print(f"[INFO] Node type distribution: {dict(Counter(type_by_id.values()))}")
print(f"[INFO] Final edge count: {len(edges)}")


# %%
# =============================================================================
# Cell 6. Assemble the AS-IS graph (typed nodes + lanes + grades + edges)
# =============================================================================
def build_asis_nodes() -> list[dict]:
    out = []
    for u in units:
        out.append({
            "id": u["id"],
            "name": u["name"],
            "lane": u.get("actor", MISSING),      # organizational swimlane
            "type": type_by_id[u["id"]],           # shape type
            "grade": u.get("auto_grade", MISSING), # automatability grade
            "rationale": u.get("auto_rationale", ""),
        })
    return out


asis_nodes = build_asis_nodes()

# Human-touched activity = any node whose work is performed by a person.
# Convention: a node is "AI-performed" in AS-IS only if the interview already
# attributes it to a system/GPT; everything else is human in AS-IS.
def is_ai_actor(node: dict) -> bool:
    a = (node.get("lane") or "").lower()
    s = ""  # system field not carried into the graph node; lane is the signal
    return a in {"system", "gpt", "ai", "ai agent"} or "gpt" in a

asis_human_nodes = [n for n in asis_nodes if not is_ai_actor(n)]
print(f"[INFO] AS-IS total nodes        : {len(asis_nodes)}")
print(f"[INFO] AS-IS human-touched nodes: {len(asis_human_nodes)}")


# %%
# =============================================================================
# Cell 7. Derive the TO-BE graph by applying the automatability rules
#
# full    -> node moves to the AI lane; human_effort_factor = 0.0
# partial -> node splits: an AI-draft node (AI lane) + a human-review node
#            (original lane) with human_effort_factor = PARTIAL_RETAIN
# manual  -> unchanged; human_effort_factor = 1.0
#
# We record a per-node "human_effort_factor" so the effort agent (nb 02) can
# multiply it by the estimated minutes to get TO-BE human effort directly.
# =============================================================================
PARTIAL_RETAIN = 0.30          # residual human effort for partial tasks (parameter)
AI_LANE = "AI Agent"           # single synthetic lane for AI-performed work

def derive_tobe(asis_nodes: list[dict]) -> tuple[list[dict], list[dict]]:
    tobe_nodes: list[dict] = []
    id_map: dict[str, list[str]] = {}   # asis id -> resulting tobe ids (for edges)

    for n in asis_nodes:
        g = n.get("grade", MISSING)
        base = {
            "src_id": n["id"], "name": n["name"], "type": n["type"],
            "grade": g, "rationale": n.get("rationale", ""),
        }
        if g == "full":
            nid = n["id"] + "_ai"
            tobe_nodes.append({**base, "id": nid, "lane": AI_LANE,
                               "human_effort_factor": 0.0, "role": "ai"})
            id_map[n["id"]] = [nid]
        elif g == "partial":
            ai_id = n["id"] + "_ai"
            hr_id = n["id"] + "_review"
            tobe_nodes.append({**base, "id": ai_id, "lane": AI_LANE,
                               "name": n["name"] + " (AI draft)",
                               "human_effort_factor": 0.0, "role": "ai"})
            tobe_nodes.append({**base, "id": hr_id, "lane": n["lane"],
                               "name": n["name"] + " (human review)",
                               "human_effort_factor": PARTIAL_RETAIN, "role": "human"})
            id_map[n["id"]] = [ai_id, hr_id]
        else:  # manual (or MISSING treated conservatively as manual)
            nid = n["id"] + "_m"
            tobe_nodes.append({**base, "id": nid, "lane": n["lane"],
                               "human_effort_factor": 1.0, "role": "human"})
            id_map[n["id"]] = [nid]
    return tobe_nodes, id_map


def remap_edges(edges: list[dict], id_map: dict[str, list[str]]) -> list[dict]:
    """Reconnect edges through the TO-BE id expansion. For a split (partial)
    node, flow enters the AI-draft node and exits the human-review node."""
    def entry(src): return id_map[src][0]          # first resulting node
    def exit_(src): return id_map[src][-1]          # last resulting node
    out = []
    for e in edges:
        if e["from"] in id_map and e["to"] in id_map:
            out.append({"from": exit_(e["from"]), "to": entry(e["to"]),
                        "label": e.get("label", "")})
    # Also stitch the internal AI->review edge for each split node.
    for src, ids in id_map.items():
        if len(ids) == 2:
            out.append({"from": ids[0], "to": ids[1], "label": ""})
    return out


tobe_nodes, id_map = derive_tobe(asis_nodes)
tobe_edges = remap_edges(edges, id_map)
tobe_human_nodes = [n for n in tobe_nodes if n["role"] == "human"]

# --- Three distinct reduction metrics (the paper reports all three) ----------
# Metric A: raw human-touched NODE COUNT (undercounts partial reductions,
#           because each 'partial' task retains one human-review node).
# Metric B: human EFFORT (time-weighted) — computed in nb 02 once Agent 2 has
#           estimated minutes; this is the primary ROI metric.
# Metric C: fully-eliminated human activities (grade 'full' & human in AS-IS)
#           vs. partially reduced ones — closest to the author's headline count.
asis_human_full = [n for n in asis_nodes
                   if not is_ai_actor(n) and n.get("grade") == "full"]
asis_human_partial = [n for n in asis_nodes
                      if not is_ai_actor(n) and n.get("grade") == "partial"]

print(f"[INFO] TO-BE total nodes        : {len(tobe_nodes)}")
print(f"[INFO] TO-BE human-touched nodes: {len(tobe_human_nodes)}")
print("[INFO] --- Reduction metrics ---")
print(f"[Metric A] Human node count : {len(asis_human_nodes)} -> "
      f"{len(tobe_human_nodes)} "
      f"({(1 - len(tobe_human_nodes)/max(len(asis_human_nodes),1))*100:.0f}% down)")
print(f"[Metric C] Fully-eliminated human activities: {len(asis_human_full)}; "
      f"partially reduced: {len(asis_human_partial)}")
print("[Metric B] Human EFFORT (time-weighted) -> computed in nb 02 (Agent 2).")


# %%
# =============================================================================
# Cell 8. Persist the process graph (AS-IS + TO-BE) for effort & figure stages
# =============================================================================
GRAPH_PATH = INFER / "process_graph.json"
graph_obj = {
    "agent": "agent1_5_process_graph",
    "source": rel(A1_PATH),
    "params": {"partial_retain": PARTIAL_RETAIN, "ai_lane": AI_LANE},
    "lane_order": lane_order,
    "as_is": {"nodes": asis_nodes, "edges": edges},
    "to_be": {"nodes": tobe_nodes, "edges": tobe_edges},
    "counts": {
        "asis_total": len(asis_nodes),
        "asis_human": len(asis_human_nodes),
        "tobe_total": len(tobe_nodes),
        "tobe_human": len(tobe_human_nodes),
        # Metric A: human-node-count reduction (%).
        "node_count_reduction_pct": round(
            (1 - len(tobe_human_nodes) / max(len(asis_human_nodes), 1)) * 100, 1),
        # Metric C: activities fully vs. partially freed of human effort.
        "fully_eliminated_human": len(asis_human_full),
        "partially_reduced_human": len(asis_human_partial),
        # Metric B (effort/time reduction %) is filled downstream by Agent 2.
        "effort_reduction_pct": None,
    },
}
GRAPH_PATH.write_text(json.dumps(graph_obj, ensure_ascii=False, indent=2),
                      encoding="utf-8")
print(f"[INFO] Process graph -> {rel(GRAPH_PATH)}")

# Paper table: per-node AS-IS/TO-BE mapping with grades and effort factors.
rows = []
for n in asis_nodes:
    for tid in id_map[n["id"]]:
        tn = next(t for t in tobe_nodes if t["id"] == tid)
        rows.append({
            "asis_id": n["id"], "name": n["name"], "lane": n["lane"],
            "type": n["type"], "grade": n["grade"],
            "tobe_id": tn["id"], "tobe_lane": tn["lane"],
            "role": tn["role"], "human_effort_factor": tn["human_effort_factor"],
        })
map_df = pd.DataFrame(rows)
map_csv = TAB / "asis_tobe_mapping.csv"
map_df.to_csv(map_csv, index=False, encoding="utf-8-sig")
print(f"[INFO] AS-IS/TO-BE mapping table -> {rel(map_csv)}")
print(f"[INFO] Agent-1.5 spend this run: ${COST.total_usd():.5f}")

[INFO] CostTracker, llm_call() re-established for nb 015.
[INFO] Loaded 46 work units from artifacts\inference\agent1_work_units.json
[INFO] Derived 9 organizational lanes: ['TCB business team', 'system', 'ECN', 'inspector', 'evaluator', 'evaluation support team', 'evaluation team', 'reviewer', 'evaluation team leader']
[INFO] Agent-1.5 prompt ready (system 1379 chars, user 5281 chars).
[INFO] Agent 1.5 returned 46 nodes, 47 edges (cached=True, cost=$0.00124).
[INFO] Node type distribution: {'start': 1, 'task': 41, 'decision': 2, 'end': 2}
[INFO] Final edge count: 47
[INFO] AS-IS total nodes        : 46
[INFO] AS-IS human-touched nodes: 35
[INFO] TO-BE total nodes        : 68
[INFO] TO-BE human-touched nodes: 30
[INFO] --- Reduction metrics ---
[Metric A] Human node count : 35 -> 30 (14% down)
[Metric C] Fully-eliminated human activities: 5; partially reduced: 22
[Metric B] Human EFFORT (time-weighted) -> computed in nb 02 (Agent 2).
[INFO] Process graph -> artifacts\inference\process_